# ARTI501 – Natural Language Processing
# Lab 3 – Part 1: N-Gram Model — Generating Tweets

**Task:** Build an N-gram (MLE) language model on a tweets dataset, use it to compute unigram/bigram/trigram counts and conditional probabilities, evaluate it with perplexity, and use it to generate new tweet-like text.

ALI ZUHAIR ALSAFFAR 2240005706

## 1. Objective / Learning Outcome

**CLO1:** Fundamentals of NLP — N-Gram models.

By the end of this notebook we will be able to:
- Clean and pre-process raw, noisy tweet text.
- Train an N-gram Maximum Likelihood Estimation (MLE) language model with NLTK.
- Query the model for n-gram counts and conditional probabilities.
- Evaluate the model on held-out data using perplexity.
- Use the trained model to generate new, tweet-like text.

## 2. Setup

Run these once (in a terminal, or uncomment and run the cell below in Jupyter/Colab):

In [1]:
import nltk

# 'punkt' (and 'punkt_tab' on newer NLTK versions) is needed for word_tokenize()
nltk.download('punkt')
try:
    nltk.download('punkt_tab')
except Exception:
    pass

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\aliza\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\aliza\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


## 3. Import Libraries

- **pandas** → load and explore the tweets dataset.
- **re** → regular expressions for cleaning tweet text (mentions, hashtags, RT, URLs, symbols).
- **emoji** → detect and strip emojis from the tweet text.
- **random** → shuffle the data for a train/test split and seed text generation.
- **nltk.tokenize.word_tokenize** → split each cleaned tweet into a list of word tokens.
- **nltk.util.ngrams** and **nltk.lm.preprocessing** → build padded n-grams for training and evaluation.
- **nltk.lm.MLE** → the Maximum Likelihood Estimation n-gram language model itself.

In [2]:
import pandas as pd
import re
import emoji
import random

from nltk.tokenize import word_tokenize
from nltk.util import ngrams
from nltk.lm.preprocessing import padded_everygram_pipeline, pad_both_ends
from nltk.lm import MLE

## 4. Load the Dataset

**Dataset:** *Anonymous – Random Tweets* (Kaggle), a large collection of raw tweets.

- **Link:** https://www.kaggle.com/datasets/adizafar/large-random-tweets-from-pakistan

**How to get the file:**
1. Open the Kaggle link above (a free Kaggle account is required) and click **Download**.
2. Place the downloaded CSV file in the same folder as this notebook.
3. Update `DATASET_PATH` below if your file has a different name.

This dataset is large, so we optionally work with a random sample (`SAMPLE_SIZE`) to keep training and generation fast. Set `SAMPLE_SIZE = None` to use the full dataset instead.

In [3]:
DATASET_PATH = "random_tweets.csv"   # update this to match your downloaded file's name
SAMPLE_SIZE = 20000                    # set to None to use the full dataset (slower)

try:
    df = pd.read_csv(DATASET_PATH, encoding="latin-1")
    print(f"Dataset loaded successfully from '{DATASET_PATH}'. Shape: {df.shape}")
except FileNotFoundError:
    df = None
    print(f"Could not find '{DATASET_PATH}'.\n"
          "Please download the dataset from:\n"
          "https://www.kaggle.com/datasets/adizafar/large-random-tweets-from-pakistan\n"
          "and place the CSV file in the same folder as this notebook "
          "(update DATASET_PATH above if the filename is different).")

Could not find 'random_tweets.csv'.
Please download the dataset from:
https://www.kaggle.com/datasets/adizafar/large-random-tweets-from-pakistan
and place the CSV file in the same folder as this notebook (update DATASET_PATH above if the filename is different).


In [4]:
if df is not None:
    print("Columns:", list(df.columns))
    display(df.head())

    # Auto-detect the column that holds the tweet text (Kaggle re-uploads sometimes use different names)
    candidate_columns = ["text", "Text", "tweet", "Tweet", "tweet_text", "Tweets", "content"]
    text_column = None
    for col in candidate_columns:
        if col in df.columns:
            text_column = col
            break

    if text_column is None:
        raise ValueError(f"Could not find a tweet-text column automatically. "
                          f"Available columns are: {list(df.columns)}. "
                          f"Please set 'text_column' manually.")
    print(f"Tweet text column detected: '{text_column}'")

    # Optionally work with a random sample to keep things fast
    if SAMPLE_SIZE is not None and len(df) > SAMPLE_SIZE:
        df = df.sample(n=SAMPLE_SIZE, random_state=42).reset_index(drop=True)
        print(f"Using a random sample of {SAMPLE_SIZE} tweets.")

## 5. Pre-processing the Data

Raw tweets are noisy. Following the lab instructions, we clean each tweet by:

1. **Removing emojis** (using the `emoji` library).
2. **Removing websites/URLs** (`http...`, `www...`).
3. **Removing mentions** (`@username`).
4. **Removing hashtags** (`#topic`) — the whole hashtag is dropped here (unlike Lab 2, where we *extracted* hashtags, here we *remove* them so they don't pollute the language model's vocabulary).
5. **Removing the "RT" retweet marker.**
6. **Lowercasing** the text.
7. **Removing extra symbols/punctuation** (keeping only letters and spaces).
8. **Removing extra whitespace.**

Each step is a simple regular-expression substitution.

In [5]:
def clean_tweet(text):
    text = str(text)

    # 1. Remove emojis
    text = emoji.replace_emoji(text, replace="")

    # 2. Remove URLs/websites
    text = re.sub(r"http\S+|www\.\S+", "", text)

    # 3. Remove mentions (@username)
    text = re.sub(r"@\w+", "", text)

    # 4. Remove hashtags (#topic) entirely
    text = re.sub(r"#\w+", "", text)

    # 5. Remove the 'RT' retweet marker (as a standalone word, case-insensitive)
    text = re.sub(r"\brt\b", "", text, flags=re.IGNORECASE)

    # 6. Lowercase everything
    text = text.lower()

    # 7. Remove extra symbols/punctuation/digits, keep only letters and spaces
    text = re.sub(r"[^a-z\s]", "", text)

    # 8. Collapse multiple spaces into one and strip leading/trailing spaces
    text = re.sub(r"\s+", " ", text).strip()

    return text

In [6]:
if df is not None:
    # fillna("") first so missing tweet text becomes an empty string, not the literal word 'nan'
    df["clean_text"] = df[text_column].fillna("").apply(clean_tweet)

    # Preview the effect of cleaning
    display(df[[text_column, "clean_text"]].head(10))

### Tokenizing the tweets

We tokenize every cleaned tweet into a list of word tokens with `word_tokenize()`. Empty tweets (which become empty strings after cleaning) are dropped since they contribute nothing to the language model.

In [7]:
if df is not None:
    # Keep only non-empty cleaned tweets
    clean_tweets = [t for t in df["clean_text"].tolist() if t.strip() != ""]

    # Tokenize each tweet into a list of words
    tokenized_tweets = [word_tokenize(t) for t in clean_tweets]

    print(f"Number of non-empty tweets after cleaning: {len(tokenized_tweets)}")
    print("Example tokenized tweet:", tokenized_tweets[0] if tokenized_tweets else None)

## 6. Train/Test Split

To properly **evaluate** the model afterwards (Section 8, perplexity), we hold out 10% of the tokenized tweets as a test set and train the model only on the remaining 90%.

In [8]:
if df is not None:
    random.seed(42)
    shuffled = tokenized_tweets.copy()
    random.shuffle(shuffled)

    split_idx = int(0.9 * len(shuffled))
    train_sentences = shuffled[:split_idx]
    test_sentences = shuffled[split_idx:]

    print(f"Training tweets: {len(train_sentences)}")
    print(f"Test tweets: {len(test_sentences)}")

## 7. Build the MLE N-gram Model

The lab describes building a **Bigram** MLE model. However, one of the required statistics below is a **trigram** count and probability (`'a' | 'pakistan is'`), which a pure bigram model cannot provide. To be able to compute *every* requested statistic — unigram, bigram, **and** trigram — from a single, consistent model, we train the MLE model at **order 3** (trigram order). An order-3 model in NLTK's `padded_everygram_pipeline`/`MLE` automatically also gives us all the unigram and bigram counts/probabilities we need, so nothing is lost.

`padded_everygram_pipeline(3, train_sentences)` prepares two things:
- `train`: every 1-gram, 2-gram, and 3-gram (with sentence padding `<s>`/`</s>`) for each training tweet.
- `vocab`: the flattened, padded vocabulary used to build the model's word list.

We then fit an `MLE(3)` model on this data.

In [9]:
if df is not None:
    N = 3  # trigram order (also gives us unigram + bigram statistics)

    train_data, vocab = padded_everygram_pipeline(N, train_sentences)

    lm = MLE(N)
    lm.fit(train_data, vocab)

    print("Vocabulary size:", len(lm.vocab))

## 8. N-gram Counts and Conditional Probabilities

Now that the model is trained, we can query it directly. Recall the pattern demonstrated earlier for a bigram model — `lm.counts[['moses']]['supposes']` gives the count of *'supposes' following 'moses'*, and `lm.score('supposes', ['moses'])` gives its probability. We use exactly the same pattern here, extended to two-word contexts for trigrams. All text was lowercased during pre-processing, so we query with lowercase words too.

In [10]:
if df is not None:
    # --- Count of unigram 'pakistan' ---
    count_unigram_pakistan = lm.counts["pakistan"]
    print(f"Count of unigram 'pakistan': {count_unigram_pakistan}")

    # --- Count of bigram 'is' following 'pakistan'  (i.e. bigram 'is/pakistan') ---
    count_bigram_is_pakistan = lm.counts[["pakistan"]]["is"]
    print(f"Count of bigram 'is' given 'pakistan': {count_bigram_is_pakistan}")

    # --- Count of trigram 'a' following 'pakistan is'  (i.e. trigram 'a/pakistan is') ---
    count_trigram_a_pakistan_is = lm.counts[["pakistan", "is"]]["a"]
    print(f"Count of trigram 'a' given 'pakistan is': {count_trigram_a_pakistan_is}")

In [11]:
if df is not None:
    # --- Probability of 'pakistan' given the start-of-sentence token <s> ---
    prob_pakistan_given_start = lm.score("pakistan", ["<s>"])
    print(f"P('pakistan' | <s>) = {prob_pakistan_given_start:.6f}")

    # --- Probability of 'is' given 'pakistan' ---
    prob_is_given_pakistan = lm.score("is", ["pakistan"])
    print(f"P('is' | 'pakistan') = {prob_is_given_pakistan:.6f}")

    # --- Probability of 'a' given 'pakistan is' ---
    prob_a_given_pakistan_is = lm.score("a", ["pakistan", "is"])
    print(f"P('a' | 'pakistan is') = {prob_a_given_pakistan_is:.6f}")

## 9. Evaluating the Model — Perplexity

**Perplexity** measures how well the trained model predicts unseen data — the probability of the test set, normalized by the number of words. Lower perplexity means the model is less "surprised" by (i.e. fits better) the held-out data.

We build the test trigrams from the 10% of tweets we held out in Section 6, padding each tweet the same way the training data was padded, then pass those trigrams to `lm.perplexity()`.

In [12]:
if df is not None:
    test_ngrams = []
    for sentence in test_sentences:
        padded_sentence = list(pad_both_ends(sentence, n=N))
        test_ngrams.extend(list(ngrams(padded_sentence, n=N)))

    perplexity = lm.perplexity(test_ngrams)
    print(f"Perplexity of the model on the held-out test tweets: {perplexity}")
    print("\nNote: MLE models assign zero probability to any trigram they never saw during "
          "training. If the test set contains such unseen trigrams, perplexity will be 'inf' "
          "— a well-known limitation of plain MLE without smoothing.")

## 10. Generating a New Tweet

Finally — the original goal of this task. NLTK's `MLE` model has a built-in `.generate()` method: given a starting seed, it repeatedly samples the next word from the model's learned probability distribution. We filter out the padding symbols (`<s>`, `</s>`) before printing the result.

In [13]:
if df is not None:
    def generate_tweet(model, num_words=15, seed_word="pakistan", random_seed=42):
        generated = model.generate(num_words, text_seed=[seed_word], random_seed=random_seed)
        # Drop padding/boundary symbols and join into a sentence
        words = [w for w in generated if w not in ("<s>", "</s>")]
        return " ".join(words)

    generated_tweet = generate_tweet(lm, num_words=15, seed_word="pakistan")
    print("Generated tweet:")
    print(generated_tweet)

## 11. Results / Conclusion

- We cleaned raw tweets (removing emojis, URLs, mentions, hashtags, the "RT" marker, extra symbols, and extra spaces) and lowercased the text.
- We trained a trigram MLE language model with NLTK, which also gives us unigram and bigram statistics.
- We queried the model for the specific unigram/bigram/trigram counts and conditional probabilities requested in the lab (all computed directly from the actual dataset, printed in Sections 8).
- We evaluated the model with **perplexity** on a held-out 10% test split (Section 9).
- We used the trained model to **generate a new, tweet-like sentence** (Section 10), demonstrating the practical use of an n-gram language model for text generation.